# Диагностика: trx_cnt / trx_sum Excel vs Lake — май 2026

## Зачем
В `row match` за **2026-05** `trx_cnt`/`trx_sum` ~55% exact, при этом:
- keys / `retl_cnt` / `term_cnt` ≈ 100%
- `commission_monthly` ≈ 99.6%
- Jan–Apr и **июнь** по trx ≈ 99.5%+

Нужно понять: косяк **Excel мая**, косяк **агрегации compare**, или косяк **озера (section 05)**.

## Что делает тетрадка
1. Грузит Excel мая (+ июнь как контроль) и `final_df` из period CSV / checkpoint.
2. Повторяет ту же нормализацию ключей и агрегацию, что в `01_07_acq_dash_jan_jun_mpos`.
3. Считает:
   - resolved-колонки Excel
   - дубли `(inn, agr)` в сыром Excel
   - итоги lake vs excel + ratio
   - exact match % включая `commission_from_ops` (тот же section 05)
   - TOP расхождения + распределение ratio lake/excel
   - эффект `sum` vs `max` для `trx_sum` в Excel (подозрение на дубли)
4. Сравнивает паттерн мая с июнем.

## Входы (пути как в боевом ноутбуке)
- Excel: `/home/jovyan/documents/Equaring/Data/05_Май_2026.xlsx`
- Lake: `final_df_period_2026_01_2026_06_mpos.csv` или checkpoint месяца


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 80)

# --- config ---
TARGET_MONTH = '2026-05'
CONTROL_MONTH = '2026-06'  # контроль: trx там сходятся

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')

excel_by_month = {
    '2026-05': DATA_DIR / '05_Май_2026.xlsx',
    '2026-06': DATA_DIR / '06_Июнь_2026.xlsx',
}
# header как в боевом ноутбуке (май/июнь = 0; янв/фев = 1)
excel_header_by_month = {
    '2026-01': 1,
    '2026-02': 1,
}

period_csv = DATA_DIR / 'final_df_period_2026_01_2026_06_mpos.csv'
checkpoint_dir = DATA_DIR / 'checkpoints_final_df_2026_01_2026_06_mpos'

OUT_DIR = DATA_DIR / 'debug_trx_may_2026'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('TARGET_MONTH =', TARGET_MONTH)
print('CONTROL_MONTH =', CONTROL_MONTH)
print('period_csv exists =', period_csv.exists(), period_csv)
print('checkpoint_dir exists =', checkpoint_dir.exists(), checkpoint_dir)
for m, p in excel_by_month.items():
    print(f'Excel {m}: exists={p.exists()} | {p}')


## Helpers (как в боевом compare)


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda x: re.sub(r'\s+', ' ', str(x).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def exact_match(a, b, atol=0.0):
    a = pd.to_numeric(a, errors='coerce')
    b = pd.to_numeric(b, errors='coerce')
    both_na = a.isna() & b.isna()
    if atol and atol > 0:
        close = (a - b).abs() <= atol
    else:
        close = a == b
    return both_na | close


COL_MAP = {
    'inn_col': ['ИНН', 'inn', 'c_inn'],
    'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
    'retl_col': ['Кол-во торговых точек', 'Ко-во торговых точек', 'Количество торговых точек'],
    'term_col': ['Кол-во терминалов', 'Количество терминалов'],
    'trx_cnt_col': ['Количество операций', 'Количеств операций', 'trx_cnt'],
    'trx_sum_col': ['Сумма операций', 'Сумма опреаций', 'trx_sum'],
    'comm_ops_col': [
        'Комиссия эквайринга',
        'Комиссия (% с операций)',
        'Комиссия \n(% с операций)',
        'Комиссия % с операций',
    ],
    'comm_monthly_col': [
        'Комиссия в месяц',
        'Комиссия CN (₽ в месяц)',
        'Комиссия (₽ в месяц)',
        'Комиссия \n(₽ в месяц)',
        'Комиссия (руб в месяц)',
    ],
}


## 1) Загрузка lake `final_df` за месяц


In [ ]:
def load_final_df_month(month_label: str) -> pd.DataFrame:
    # 1) period CSV
    if period_csv.exists():
        df = pd.read_csv(period_csv, dtype={'inn': 'string', 'agr_id': 'string', 'n_agr': 'string'})
        if 'report_month' not in df.columns:
            raise RuntimeError(f'no report_month in {period_csv}')
        out = df.loc[df['report_month'].astype(str) == month_label].copy()
        if len(out):
            print(f'lake from period CSV: {month_label} rows={len(out):,}')
            return out
        print(f'period CSV has 0 rows for {month_label}, try checkpoint')

    # 2) checkpoint
    month_us = month_label.replace('-', '_')
    candidates = []
    if checkpoint_dir.exists():
        candidates += sorted(checkpoint_dir.glob(f'*{month_us}*'))
        candidates += sorted(checkpoint_dir.glob(f'*{month_label}*'))
        for pat in [
            f'final_df_{month_label}.csv',
            f'final_df_{month_label}.parquet',
            f'{month_label}.csv',
            f'{month_label}.parquet',
        ]:
            p = checkpoint_dir / pat
            if p.exists():
                candidates = [p] + candidates
        if not candidates:
            print('checkpoint files:', [p.name for p in sorted(checkpoint_dir.iterdir())[:40]])

    for p in candidates:
        if not p.exists() or not p.is_file():
            continue
        if p.suffix.lower() == '.csv':
            out = pd.read_csv(p, dtype={'inn': 'string', 'agr_id': 'string', 'n_agr': 'string'})
        elif p.suffix.lower() == '.parquet':
            out = pd.read_parquet(p)
        else:
            continue
        if 'report_month' in out.columns:
            out = out.loc[out['report_month'].astype(str) == month_label].copy()
        print(f'lake from checkpoint: {p.name} rows={len(out):,}')
        return out

    raise FileNotFoundError(
        f'Не найден final_df за {month_label}. '
        f'Проверь {period_csv} или {checkpoint_dir}'
    )


def build_lake_agg(final_df: pd.DataFrame) -> pd.DataFrame:
    lk = final_df.copy()
    if 'agr_id' not in lk.columns and 'n_agr' in lk.columns:
        lk['agr_id'] = lk['n_agr']
    lk['inn_key'] = lk['inn'].apply(normalize_inn_q1)
    lk['agr_id_key'] = lk['agr_id'].apply(normalize_agr_q1)

    for c in ['trx_cnt', 'trx_sum', 'commission_from_ops', 'commission_monthly', 'retl_cnt', 'term_cnt']:
        if c not in lk.columns:
            lk[c] = np.nan
        lk[c] = pd.to_numeric(lk[c], errors='coerce')

    agg = (
        lk.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['inn_key', 'agr_id_key'], as_index=False)
          .agg(
              trx_cnt_lake=('trx_cnt', 'max'),
              trx_sum_lake=('trx_sum', 'max'),
              commission_from_ops_lake=('commission_from_ops', 'max'),
              commission_monthly_lake=('commission_monthly', 'max'),
              retl_cnt_lake=('retl_cnt', 'max'),
              term_cnt_lake=('term_cnt', 'max'),
              rows_lake=('trx_cnt', 'size'),
          )
    )
    return agg


lake_raw = load_final_df_month(TARGET_MONTH)
print('lake columns sample:', list(lake_raw.columns)[:40])
lake_agg = build_lake_agg(lake_raw)
print('lake_agg rows =', f'{len(lake_agg):,}', '| unique inn =', lake_agg['inn_key'].nunique())
display(lake_agg.head(5))


## 2) Excel: колонки, дубли, агрегации (`sum` vs `max`)


In [ ]:
def load_excel_raw(month_label: str):
    path = excel_by_month[month_label]
    header = int(excel_header_by_month.get(month_label, 0))
    ex = pd.read_excel(path, header=header)
    resolved = {k: pick_col_robust(ex.columns, v) for k, v in COL_MAP.items()}
    print(f'=== Excel {month_label} ===')
    print('path =', path)
    print('header =', header, '| shape =', ex.shape)
    print('resolved columns:')
    for k, v in resolved.items():
        print(f'  {k}: {v!r}')
    missing = [k for k, v in resolved.items() if v is None]
    if missing:
        print('MISSING:', missing)
        print('available columns:', list(ex.columns))
    return ex, resolved, header


def enrich_excel(ex, resolved):
    out = ex.copy()
    out['inn_key'] = out[resolved['inn_col']].apply(normalize_inn_q1)
    out['agr_id_key'] = out[resolved['agr_col']].apply(normalize_agr_q1)
    out['trx_cnt_excel'] = pd.to_numeric(out[resolved['trx_cnt_col']], errors='coerce')
    out['trx_sum_excel'] = to_num_series(out[resolved['trx_sum_col']])
    out['commission_from_ops_excel'] = to_num_series(out[resolved['comm_ops_col']])
    out['commission_monthly_excel'] = to_num_series(out[resolved['comm_monthly_col']])
    if resolved.get('retl_col'):
        out['retl_cnt_excel'] = pd.to_numeric(out[resolved['retl_col']], errors='coerce')
    if resolved.get('term_col'):
        out['term_cnt_excel'] = pd.to_numeric(out[resolved['term_col']], errors='coerce')
    return out


def excel_dup_stats(ex_enr: pd.DataFrame) -> pd.DataFrame:
    g = (
        ex_enr.dropna(subset=['inn_key', 'agr_id_key'])
        .groupby(['inn_key', 'agr_id_key'], as_index=False)
        .agg(
            rows=('trx_cnt_excel', 'size'),
            trx_cnt_nunique=('trx_cnt_excel', 'nunique'),
            trx_sum_nunique=('trx_sum_excel', 'nunique'),
            trx_cnt_max=('trx_cnt_excel', 'max'),
            trx_sum_max=('trx_sum_excel', 'max'),
            trx_sum_sum=('trx_sum_excel', 'sum'),
            trx_cnt_sum=('trx_cnt_excel', 'sum'),
        )
    )
    g['sum_vs_max_trx_sum_ratio'] = np.where(
        g['trx_sum_max'].fillna(0) == 0,
        np.nan,
        g['trx_sum_sum'] / g['trx_sum_max'],
    )
    return g


ex_raw, ex_resolved, ex_header = load_excel_raw(TARGET_MONTH)
ex_enr = enrich_excel(ex_raw, ex_resolved)
ex_dups = excel_dup_stats(ex_enr)

print('\n=== Excel key / duplicate profile ===')
print('raw rows =', f'{len(ex_enr):,}')
print('valid keys =', f"{ex_enr.dropna(subset=['inn_key', 'agr_id_key']).shape[0]:,}")
print('unique (inn,agr) =', f'{len(ex_dups):,}')
print('keys with rows>1 =', int((ex_dups['rows'] > 1).sum()))
print(
    'keys where sum(trx_sum) != max(trx_sum) =',
    int((ex_dups['trx_sum_sum'].fillna(0) != ex_dups['trx_sum_max'].fillna(0)).sum()),
)
print(
    'keys where sum/max trx_sum ≈ 2 =',
    int(((ex_dups['sum_vs_max_trx_sum_ratio'] - 2).abs() < 0.01).sum()),
)
display(ex_dups.sort_values('rows', ascending=False).head(15))

# боевая агрегация Excel (как в mpos-ноутбуке)
ex_agg_battle = (
    ex_enr.dropna(subset=['inn_key', 'agr_id_key'])
    .groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg(
        trx_cnt_excel=('trx_cnt_excel', 'max'),
        trx_sum_excel=('trx_sum_excel', 'sum'),  # battle: SUM
        commission_from_ops_excel=('commission_from_ops_excel', 'sum'),
        commission_monthly_excel=('commission_monthly_excel', 'max'),
    )
)
# альтернатива: max для trx_sum
ex_agg_max = (
    ex_enr.dropna(subset=['inn_key', 'agr_id_key'])
    .groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg(
        trx_cnt_excel=('trx_cnt_excel', 'max'),
        trx_sum_excel=('trx_sum_excel', 'max'),
        commission_from_ops_excel=('commission_from_ops_excel', 'max'),
        commission_monthly_excel=('commission_monthly_excel', 'max'),
    )
)
print(
    '\nbattle excel totals: trx_cnt=',
    ex_agg_battle['trx_cnt_excel'].fillna(0).sum(),
    'trx_sum=',
    ex_agg_battle['trx_sum_excel'].fillna(0).sum(),
)
print(
    'max-agg excel totals: trx_cnt=',
    ex_agg_max['trx_cnt_excel'].fillna(0).sum(),
    'trx_sum=',
    ex_agg_max['trx_sum_excel'].fillna(0).sum(),
)


## 3) Merge + totals + exact match (включая `commission_from_ops`)


In [ ]:
def compare_side(lake_agg, ex_agg, label: str, money_atol=0.01):
    cmp = lake_agg.merge(ex_agg, on=['inn_key', 'agr_id_key'], how='outer', indicator=True)
    both = cmp[cmp['_merge'] == 'both'].copy()
    print(f'\n===== COMPARE {label} =====')
    print(
        'keys: both=', f"{(cmp['_merge'] == 'both').sum():,}",
        '| only_lake=', f"{(cmp['_merge'] == 'left_only').sum():,}",
        '| only_excel=', f"{(cmp['_merge'] == 'right_only').sum():,}",
    )

    metrics = [
        ('trx_cnt', 'trx_cnt_lake', 'trx_cnt_excel', 0.0),
        ('trx_sum', 'trx_sum_lake', 'trx_sum_excel', money_atol),
        ('commission_from_ops', 'commission_from_ops_lake', 'commission_from_ops_excel', money_atol),
        ('commission_monthly', 'commission_monthly_lake', 'commission_monthly_excel', money_atol),
    ]
    rows = []
    for name, lc, ec, atol in metrics:
        if lc not in both.columns or ec not in both.columns:
            continue
        m = exact_match(both[lc], both[ec], atol=atol)
        exact_n = int(m.sum())
        n = len(both)
        rows.append({
            'label': label,
            'metric': name,
            'exact_n': exact_n,
            'n_both': n,
            'exact_pct': round(100.0 * exact_n / n, 2) if n else None,
            'mismatch_n': n - exact_n,
            'sum_lake': float(pd.to_numeric(both[lc], errors='coerce').fillna(0).sum()),
            'sum_excel': float(pd.to_numeric(both[ec], errors='coerce').fillna(0).sum()),
        })
        r = rows[-1]
        ratio = (r['sum_lake'] / r['sum_excel']) if r['sum_excel'] else np.nan
        print(
            f"  {name}: exact={r['exact_pct']}% ({exact_n:,}/{n:,}), mismatch={r['mismatch_n']:,} | "
            f"sum_lake={r['sum_lake']:,.2f} sum_excel={r['sum_excel']:,.2f} ratio_lake/excel={ratio:.4f}"
        )
    return cmp, both, pd.DataFrame(rows)


cmp_battle, both_battle, stats_battle = compare_side(
    lake_agg, ex_agg_battle, f'{TARGET_MONTH} battle(excel trx_sum=SUM)'
)
cmp_max, both_max, stats_max = compare_side(
    lake_agg, ex_agg_max, f'{TARGET_MONTH} alt(excel trx_sum=MAX)'
)

stats_all = pd.concat([stats_battle, stats_max], ignore_index=True)
display(stats_all)
stats_all.to_csv(OUT_DIR / f'stats_{TARGET_MONTH}.csv', index=False)
print('saved', OUT_DIR / f'stats_{TARGET_MONTH}.csv')


## 4) TOP mismatches + распределение ratio (ищем ×2 / систематику)


In [ ]:
def add_ratios(both: pd.DataFrame) -> pd.DataFrame:
    b = both.copy()
    for metric, lc, ec in [
        ('trx_cnt', 'trx_cnt_lake', 'trx_cnt_excel'),
        ('trx_sum', 'trx_sum_lake', 'trx_sum_excel'),
        ('commission_from_ops', 'commission_from_ops_lake', 'commission_from_ops_excel'),
    ]:
        a = pd.to_numeric(b[lc], errors='coerce')
        e = pd.to_numeric(b[ec], errors='coerce')
        b[f'{metric}_delta'] = a - e
        b[f'{metric}_abs_delta'] = (a - e).abs()
        b[f'{metric}_ratio'] = np.where(e.fillna(0) == 0, np.nan, a / e)
    return b


both_x = add_ratios(both_battle)
mis = both_x.loc[~exact_match(both_x['trx_cnt_lake'], both_x['trx_cnt_excel'])].copy()
print('trx_cnt mismatches =', f'{len(mis):,}')

print('\n=== TOP-30 by |trx_cnt delta| ===')
cols_show = [
    'inn_key', 'agr_id_key',
    'trx_cnt_lake', 'trx_cnt_excel', 'trx_cnt_delta', 'trx_cnt_ratio',
    'trx_sum_lake', 'trx_sum_excel', 'trx_sum_delta', 'trx_sum_ratio',
    'commission_from_ops_lake', 'commission_from_ops_excel',
    'commission_monthly_lake', 'commission_monthly_excel',
]
display(mis.sort_values('trx_cnt_abs_delta', ascending=False)[cols_show].head(30))

print('\n=== ratio lake/excel buckets (trx_cnt mismatches) ===')
ratio = mis['trx_cnt_ratio'].replace([np.inf, -np.inf], np.nan).dropna()
buckets = pd.cut(
    ratio,
    bins=[0, 0.4, 0.6, 0.9, 1.1, 1.5, 2.1, 3.1, 10, 1000],
    include_lowest=True,
)
display(buckets.value_counts(dropna=False).sort_index())

print('\nshare near ×2 (1.9..2.1):', float(((ratio >= 1.9) & (ratio <= 2.1)).mean()) if len(ratio) else None)
print('share near ×0.5 (0.45..0.55):', float(((ratio >= 0.45) & (ratio <= 0.55)).mean()) if len(ratio) else None)
print('median ratio:', float(ratio.median()) if len(ratio) else None)

ops_ok = exact_match(mis['commission_from_ops_lake'], mis['commission_from_ops_excel'], atol=0.01)
print('\nAmong trx_cnt mismatches:')
print('  commission_from_ops exact:', int(ops_ok.sum()), '/', len(mis))
print(
    '  commission_monthly exact:',
    int(exact_match(mis['commission_monthly_lake'], mis['commission_monthly_excel'], atol=0.01).sum()),
    '/',
    len(mis),
)

mis.sort_values('trx_cnt_abs_delta', ascending=False)[cols_show].head(200).to_csv(
    OUT_DIR / f'top_trx_mismatch_{TARGET_MONTH}.csv', index=False
)
print('saved', OUT_DIR / f'top_trx_mismatch_{TARGET_MONTH}.csv')


## 5) Контроль: тот же разбор для июня (ожидаем ~99% trx)


In [ ]:
# June lake + excel
lake_raw_ctrl = load_final_df_month(CONTROL_MONTH)
lake_agg_ctrl = build_lake_agg(lake_raw_ctrl)

ex_raw_c, ex_res_c, _ = load_excel_raw(CONTROL_MONTH)
ex_enr_c = enrich_excel(ex_raw_c, ex_res_c)
ex_dups_c = excel_dup_stats(ex_enr_c)
print('\nCONTROL excel dups rows>1:', int((ex_dups_c['rows'] > 1).sum()), '/', len(ex_dups_c))

ex_agg_c = (
    ex_enr_c.dropna(subset=['inn_key', 'agr_id_key'])
    .groupby(['inn_key', 'agr_id_key'], as_index=False)
    .agg(
        trx_cnt_excel=('trx_cnt_excel', 'max'),
        trx_sum_excel=('trx_sum_excel', 'sum'),
        commission_from_ops_excel=('commission_from_ops_excel', 'sum'),
        commission_monthly_excel=('commission_monthly_excel', 'max'),
    )
)
_, both_c, stats_c = compare_side(lake_agg_ctrl, ex_agg_c, f'{CONTROL_MONTH} battle')
display(stats_c)

side = stats_battle[['metric', 'exact_pct', 'sum_lake', 'sum_excel']].merge(
    stats_c[['metric', 'exact_pct', 'sum_lake', 'sum_excel']],
    on='metric',
    suffixes=('_may', '_jun'),
)
print('\n=== May vs June exact_pct ===')
display(side)
side.to_csv(OUT_DIR / 'may_vs_june_exact_pct.csv', index=False)


## 6) Вердикт-подсказки (авто)


In [ ]:
def verdict():
    s = stats_battle.set_index('metric')
    trx_pct = s.loc['trx_cnt', 'exact_pct'] if 'trx_cnt' in s.index else None
    ops_pct = s.loc['commission_from_ops', 'exact_pct'] if 'commission_from_ops' in s.index else None
    mon_pct = s.loc['commission_monthly', 'exact_pct'] if 'commission_monthly' in s.index else None
    trx_sum_battle = s.loc['trx_sum', 'exact_pct'] if 'trx_sum' in s.index else None
    trx_pct_maxagg = None
    if len(stats_max):
        sm = stats_max.set_index('metric')
        if 'trx_sum' in sm.index:
            trx_pct_maxagg = sm.loc['trx_sum', 'exact_pct']

    ratio_total = None
    if 'trx_cnt' in s.index and s.loc['trx_cnt', 'sum_excel']:
        ratio_total = s.loc['trx_cnt', 'sum_lake'] / s.loc['trx_cnt', 'sum_excel']

    dup_share = (ex_dups['rows'] > 1).mean() if len(ex_dups) else 0
    sum_ne_max = (
        (ex_dups['trx_sum_sum'].fillna(0) != ex_dups['trx_sum_max'].fillna(0)).mean()
        if len(ex_dups)
        else 0
    )

    print('=== AUTO VERDICT HINTS ===')
    print(f'trx_cnt exact (battle) = {trx_pct}%')
    print(f'commission_from_ops exact = {ops_pct}%')
    print(f'commission_monthly exact = {mon_pct}%')
    print(f'total trx_cnt ratio lake/excel = {ratio_total}')
    print(f'excel duplicate-key share = {dup_share:.2%}; share where sum!=max trx_sum = {sum_ne_max:.2%}')
    print(f'trx_sum exact if excel agg=MAX = {trx_pct_maxagg}%')

    print()
    if ops_pct is not None and trx_pct is not None:
        if ops_pct >= 95 and trx_pct < 80:
            print(
                '→ Похоже на проблему КОЛОНОК/ЗНАЧЕНИЙ операций в Excel мая: '
                'commission_from_ops (section 05) сходится, а trx_cnt/sum — нет.'
            )
        elif ops_pct < 80 and trx_pct < 80:
            print(
                '→ Расходятся и trx, и commission_from_ops → расхождение ИСТОЧНИКА trx '
                '(озеро section 05 vs то, что заложено в Excel), не только «подпись колонки».'
            )
        else:
            print('→ Смотри TOP mismatches и ratio buckets вручную.')

    if (
        sum_ne_max > 0.1
        and trx_pct_maxagg is not None
        and trx_sum_battle is not None
        and (trx_pct_maxagg - trx_sum_battle) > 10
    ):
        print(
            '→ Сильный эффект дублей Excel: battle sum(trx_sum) портит compare; '
            'попробуй max-агрегацию или почистить дубли в файле.'
        )

    if ratio_total is not None and (abs(ratio_total - 2) < 0.05 or abs(ratio_total - 0.5) < 0.05):
        print('→ Тоталы отличаются примерно в 2 раза — проверь двойной учёт / неполный месяц / YTD в Excel.')

    if len(stats_c):
        jun_trx = stats_c.set_index('metric').loc['trx_cnt', 'exact_pct']
        print(f'\nControl {CONTROL_MONTH} trx_cnt exact = {jun_trx}% (ожидаем ~99%)')
        if jun_trx and jun_trx >= 95 and trx_pct and trx_pct < 80:
            print(
                '→ Июнь ок, май плохой → почти наверняка файл/методика Excel мая, '
                'а не общий баг section 05.'
            )


verdict()
print('\nАртефакты:', OUT_DIR)


## Как читать результат

| Находка | Вывод |
|---------|--------|
| `commission_from_ops` ~99%, `trx_cnt` ~55% | Excel: кривые колонки/значения операций; озеро trx-комиссия ок |
| Оба `commission_from_ops` и `trx_*` ~55% | Разный источник/методика trx в Excel vs озеро за май |
| `sum`→`max` резко поднимает exact по `trx_sum` | Дубли строк в Excel мая |
| ratio ≈ 2 или 0.5 на тоталах/бакетах | Двойной учёт или неполный период в одном из источников |
| Июнь ~99%, май ~55% | Не общий баг пайплайна; копать `05_Май_2026.xlsx` |

После вердикта: либо заменить/пересобрать Excel мая, либо точечно править section 05 / compare-агрегацию — уже по факту.
